# The European Football Divide
**Cross-league comparison of competitive inequality (1992-2025)**

*Eric Schnitger, 2026*

---

## Key Findings

- **All five leagues show a widening gap** since 1992
- **Bundesliga** has the highest average gap (Bayern entrenched early)
- **Ligue 1** shows the most dramatic inflection (PSG takeover, 2011)
- **Premier League / La Liga** have the steepest rate of increase
- **Serie A** is the most volatile (Calciopoli, bankruptcies)
- **No league has reversed the trend** through regulation alone
       

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from _utils import (
   LEAGUES, scrape_league, enrich_dataframe,
   build_league_chart, build_comparison_chart, build_frequency_tables,
)
import pandas as pd
import numpy as np
import plotly.graph_objects as go

In [2]:
league_data = {}
for key in ["premier_league", "bundesliga", "la_liga", "ligue_1", "serie_a"]:
   config = LEAGUES[key]
   raw = scrape_league(key, cache_dir="../data")
   league_data[key] = enrich_dataframe(raw)
   print(f"  {config.name}: {len(league_data[key])} seasons")

total = sum(len(v) for v in league_data.values())
print(f"\nTotal: {total} season-records across 5 leagues")

  [PL] cache hit (0.0d old)
  Premier League: 33 seasons
  [BL] cache hit (0.0d old)
  Bundesliga: 32 seasons
  [LL] cache hit (0.0d old)
  La Liga: 31 seasons
  [L1] cache hit (0.0d old)
  Ligue 1: 31 seasons
  [SA] cache hit (0.0d old)
  Serie A: 33 seasons

Total: 160 season-records across 5 leagues


## The Gap Over Time (Normalized to 38 games)

In [3]:
fig = build_comparison_chart(
   league_data,
   metric="Gap (38-game)",
   title="The Growing Divide: Gap Between Champions and Survival (Normalized)",
)
fig

## Title-Winning Points (Normalized)

In [4]:
fig = build_comparison_chart(
   league_data,
   metric="Title Pts (38-game)",
   title="Title-Winning Points Across Europe (38-Game Equivalent)",
)
fig

## Relegation Survival Points (Normalized)

In [5]:
fig = build_comparison_chart(
   league_data,
   metric="Survival Pts (38-game)",
   title="Relegation Survival Points Across Europe (38-Game Equivalent)",
)
fig

## Points Ratio (Unit-Free)

In [6]:
fig = build_comparison_chart(
   league_data, metric="Ratio",
   title="Competitive Inequality: Champion-to-Survival Points Ratio",
)
fig.update_layout(yaxis=dict(title="Ratio (champion pts / survival pts)"))
fig

## League Summary Statistics

In [7]:
rows = []
for key, df in league_data.items():
   config = LEAGUES[key]
   row = {
       "League": config.name, "Country": config.country,
       "Seasons": len(df), "Teams": config.num_teams,
       "Avg Title (38g)":    round(df["Title Pts (38-game)"].mean(), 1),
       "Avg Survival (38g)": round(df["Survival Pts (38-game)"].mean(), 1),
       "Avg Gap (38g)":      round(df["Gap (38-game)"].mean(), 1),
       "Avg Ratio": round(df["Ratio"].mean(), 2),
       "Max Ratio": round(df["Ratio"].max(), 2),
   }
   x = np.arange(len(df)).astype(float)
   y = df["Gap (38-game)"].values.astype(float)
   mask = ~np.isnan(y)
   if mask.sum() >= 3:
       row["Gap Trend (pts/yr)"] = round(np.polyfit(x[mask], y[mask], 1)[0], 3)
   rows.append(row)

pd.DataFrame(rows).style.format({
   "Avg Ratio": "{:.2f}x", "Max Ratio": "{:.2f}x",
   "Gap Trend (pts/yr)": "{:+.3f}",
}).hide(axis="index")

League,Country,Seasons,Teams,Avg Title (38g),Avg Survival (38g),Avg Gap (38g),Avg Ratio,Max Ratio,Gap Trend (pts/yr)
Premier League,England,33,20,86.900000,38.200000,48.700000,2.30x,2.84x,+0.538
Bundesliga,Germany,32,18,82.900000,38.600000,44.300000,2.15x,2.81x,+0.945
La Liga,Spain,31,20,83.800000,39.300000,44.500000,2.14x,2.70x,+0.886
Ligue 1,France,31,18,78.900000,38.400000,40.500000,2.06x,2.57x,+0.779
Serie A,Italy,33,20,84.400000,38.000000,46.400000,2.24x,3.00x,+0.753


## Top-3 Club Concentration

In [8]:
conc = []
for key, df in league_data.items():
   config = LEAGUES[key]
   top4, _ = build_frequency_tables(df, config)
   total_slots = len(df) * 4
   share = top4.head(3)["Top 4 Finishes"].sum() / total_slots * 100
   conc.append({"League": config.name,
                "Top 3 Share (%)": round(share, 1),
                "Color": config.color})

conc_df = pd.DataFrame(conc).sort_values("Top 3 Share (%)")

fig = go.Figure(data=[go.Bar(
   x=conc_df["Top 3 Share (%)"], y=conc_df["League"], orientation="h",
   marker=dict(color=conc_df["Color"].tolist()),
   hovertemplate="<b>%{y}</b><br>%{x:.1f}% of all top-4 slots<extra></extra>",
)])
fig.update_layout(
   template="plotly_dark",
   title="Competitive Concentration: Top 3 Clubs' Share of Top-4 Finishes",
   xaxis=dict(title="% of Top-4 Slots", range=[0, 100]),
   height=400, width=900,
)
fig

## Decade-by-Decade Gap

In [9]:
rows = []
for key, df in league_data.items():
   config = LEAGUES[key]
   tmp = df.copy()
   tmp["Decade"] = (tmp["Start Year"] // 10) * 10
   tmp["Decade Label"] = tmp["Decade"].astype(str) + "s"
   for decade, group in tmp.groupby("Decade Label"):
       rows.append({
           "League": config.name, "Decade": decade,
           "Avg Gap (38g)": round(group["Gap (38-game)"].mean(), 1),
       })
decade_df = pd.DataFrame(rows)

fig = go.Figure()
for key in league_data:
   config = LEAGUES[key]
   sub = decade_df[decade_df["League"] == config.name].sort_values("Decade")
   fig.add_trace(go.Bar(
       x=sub["Decade"], y=sub["Avg Gap (38g)"],
       name=config.name, marker=dict(color=config.color, opacity=0.85),
   ))
fig.update_layout(
   template="plotly_dark",
   title="Average Gap by Decade (38-Game Normalized)",
   barmode="group",
   xaxis=dict(title="Decade"),
   yaxis=dict(title="Avg Gap (38g equivalent)"),
   height=500, width=1000,
   legend=dict(orientation="h", yanchor="bottom", y=1.02,
               xanchor="center", x=0.5),
)
fig

## Conclusions

1. **The divergence is universal.** Every top-5 European league is more stratified in 2025 than in 1992.
2. **The causes are structural.** TV revenue concentration and Champions League prize money create a feedback loop.
3. **The rate differs by league.** England and Spain accelerated fastest; Germany was already uneven; France had a single shock; Italy has been chaotic.
4. **Survival thresholds are universal.** ~33-37 points (38-game equivalent) across all leagues.
5. **No regulation has worked.** Not 50+1, not collective TV deals, not FFP.
       